# Pre-work

In [1]:
# imports

from pathlib import Path
import pandas as pd
import json
import re
import ollama

from tqdm import tqdm

# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
# import matplotlib.pyplot as plt

In [2]:
# config

tqdm.pandas()

In [3]:
# constants

ROOT_DIR = Path.cwd().parent
DATASET_DIR = ROOT_DIR / "datasets"

NOVIELLI_GOLD_STANDARD_DIR = DATASET_DIR / "novielli-gh-gold"
GH_GOLD_STANDARD = NOVIELLI_GOLD_STANDARD_DIR / "github_gold.csv"

COUTINHO_DIR = DATASET_DIR / "coutinho-sentiment-replication"
COUTINHO = COUTINHO_DIR / "dataset.json"

DEVA_INPUT_DIR = ROOT_DIR / "input" / "deva"
DEVA_OUTPUT_DIR = ROOT_DIR / "output" / "deva"

# Data Collection

## Novielli et al., 2020
The dataset used by Novielli et al. is in CSV format, which makes it straight forward to create a dataframe out of.

In [4]:
df_gold_standard_full = pd.read_csv(GH_GOLD_STANDARD, sep=';', quotechar='"', encoding='utf-8')
df_gold_standard_full.columns = df_gold_standard_full.columns.str.lower()

# strip outer quote
df_gold_standard_full['text'] = df_gold_standard_full['text'].str.rstrip('"')

# remove id column
df_gold_standard_full = df_gold_standard_full[['text', 'polarity']]

df_gold_standard_full

,text,polarity
0,No. I still see the wrong twins. * https://gi...,neutral
1,Reverted.,neutral
2,You can leave a queue while in queue ? (before...,neutral
3,Didn't look at SpellTargetRestrictions XD,positive
4,Not sure about what kind of line lengths the p...,neutral
...,...,...
7117,"Yeah, I'm capable of working around it. A set...",negative
7118,"Looks fine to me! On Jul 8, 2013, at 10:12 PM...",positive
7119,Formula updated to address this (would be nice...,positive
7120,I trust you,positive


### Test/Train split
Note that, in the paper, the GitHub Gold Standard dataset is split into a 30% test set and a 70% train set, on which DEVA (among other models) is run. To keep this replication as close as possible to the original, we also perform this partition with `scikit` and then use the test set for evaluation. Note that the paper does not mention a specific seed used for the split, so we use our own seed and mention that the results won't be 100% the same as the paper (however, the macro-averages should match closely).

In [5]:
_, df_gold_standard = train_test_split(df_gold_standard_full, test_size=0.3, random_state=42)
df_gold_standard = df_gold_standard.reset_index(drop=True)
df_gold_standard

,text,polarity
0,Guess there is a typo here: `translate`.,neutral
1,"@arturoc: If you multiply-include `gst.h`, wou...",neutral
2,Thank you Vlad. Your contribution to Mangos an...,positive
3,"@opdenkamp Hi Lars, I'm afraid that you forgot...",negative
4,"Ok, so let's be paranoid.",negative
...,...,...
2132,moved into encoders/decoders,neutral
2133,"I understand. In that case, we would need to h...",negative
2134,that is a good idea.,positive
2135,"Just out of curiosity, why `uv_buf_init` shoul...",neutral


## Coutinho et al., 2024
JSON format, so some normalization is required.

In [6]:
text = COUTINHO.read_text(encoding="utf-8")

if text.lstrip().startswith("["):
    data = json.loads(text)
else:
    try:
        data = [json.loads(line) for line in text.splitlines() if line.strip()]
    except json.JSONDecodeError:
        data = [json.loads(text)]

df_premo = pd.json_normalize(data, sep="_")

df_premo['polarity'] = df_premo['part2_aggregate_polarity'].where(
    df_premo['part2_aggregate_polarity'] != 'undefined',
    other=df_premo['discussion_polarity']
)
df_premo = df_premo[df_premo['polarity'].notna() & (df_premo['polarity'] != 'undefined')]

# normalize to same structure as gold standard dataset
df_premo = df_premo[['clean_message', 'polarity']].rename(columns={'clean_message': 'text'})

df_premo

,text,polarity
0,did you get a chance to follow up on the issue...,neutral
1,Thank you for this PR. This is an interesting...,neutral
2,", oh sorry for this, my fault, i'll be more c...",negative
3,I've added so that we can add documentation r...,neutral
4,Applications can still use or qualifiers to g...,negative
...,...,...
1786,"Hmm, looks like r737027 corresponds to version...",negative
1787,"<!-- First of all, thank you for your contribu...",neutral
1788,"there was a typo in the comment, also made a s...",neutral
1789,Fixed some typos. Added the ghost button type....,neutral


# DEVA Experiments

DEVA expects an input .txt file with the message to be analyzed and its ID. To demonstrate, this is the example input they provide with the tool:
```
58204 Yes, that's correct.
32957 Challenge accepted! :)
45982 No problem.  It was my pleasure, and thanks for all the help.
63699 Still need tests  Neutral
14350 My mistake, sorry
```

It outputs the original input, as well as three extra columns: its valence score, its arousal score, and an emotional state label derived from the combination of both.

For the purposes of this replication study, however, the benchmark reduces sentiment to a three-class polarity task (positive, negative, neutral), so only the valence score is used. This maps directly to Novielli et al. replication's $±[1,3]$ schema:
- valence > +1 -> positive
- valence < -1 -> negative
- valence = ± 1 -> neutral

The output for the example input looks as follows:
```
58204 Yes, that's correct.  -2  -3    DEPRESSED
32957 Challenge accepted! :)  2   5   EXCITED
45982 No problem.  It was my pleasure, and thanks for all the help.   4   3   EXCITED
63699 Still need tests  Neutral   1   -2    NEUTRAL (LA)
14350 My mistake, sorry   -2  1   NEG VALENCE
```

Therefore, to run DEVA on the datasets collected above, we need to generate valid input, and then parse the output into dataframes.

In [7]:
def write_deva_input(df, output_path, text_col='text'):
    """Write a DEVA-compatible input .txt file from a dataframe."""
    df_out = df[[text_col]].copy()
    df_out.to_csv(output_path, sep='\t', header=False, index=True)

def parse_deva_output(filepath):
    valences, arousals, labels = [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:                                                                         
        for line in f:                                                                                                       
            parts = line.rstrip('\n').rsplit('\t', 3)                                                                        
            if len(parts) == 4:                                                                                              
                _, valence, arousal, label = parts
                valences.append(int(valence.strip()))                                                                        
                arousals.append(int(arousal.strip()))
                labels.append(label.strip())                                                                                 
    return valences, arousals, labels

In [8]:
write_deva_input(df_gold_standard, DEVA_INPUT_DIR / "gold_standard.txt")
write_deva_input(df_premo, DEVA_INPUT_DIR / "premo.txt")

In [9]:
v, a, l = parse_deva_output(DEVA_OUTPUT_DIR / "gold_standard_out.txt")
df_gold_standard['valence'] = v

df_gold_standard.head(5)

,text,polarity,valence
0,Guess there is a typo here: `translate`.,neutral,1
1,"@arturoc: If you multiply-include `gst.h`, wou...",neutral,4
2,Thank you Vlad. Your contribution to Mangos an...,positive,3
3,"@opdenkamp Hi Lars, I'm afraid that you forgot...",negative,-3
4,"Ok, so let's be paranoid.",negative,1


In [10]:
v, a, l = parse_deva_output(DEVA_OUTPUT_DIR / "premo_out.txt")
df_premo['valence'] = v

df_premo.head(5)

,text,polarity,valence
0,did you get a chance to follow up on the issue...,neutral,1
1,Thank you for this PR. This is an interesting...,neutral,-3
2,", oh sorry for this, my fault, i'll be more c...",negative,-2
3,I've added so that we can add documentation r...,neutral,1
4,Applications can still use or qualifiers to g...,negative,-2


## Mapping DEVA results to polarity labels
As mentioned above, the following mapping is used:
- valence > +1 -> positive
- valence < -1 -> negative
- valence = ± 1 -> neutral

In [11]:
def valence_to_polarity(v):
    if v > 1:
        return 'positive'
    elif v < -1:
        return 'negative'
    else:
        return 'neutral'

In [12]:
df_gold_standard['deva_polarity'] = df_gold_standard['valence'].apply(valence_to_polarity)
df_gold_standard = df_gold_standard.drop(columns=['valence'], errors='ignore')
df_gold_standard

,text,polarity,deva_polarity
0,Guess there is a typo here: `translate`.,neutral,neutral
1,"@arturoc: If you multiply-include `gst.h`, wou...",neutral,positive
2,Thank you Vlad. Your contribution to Mangos an...,positive,positive
3,"@opdenkamp Hi Lars, I'm afraid that you forgot...",negative,negative
4,"Ok, so let's be paranoid.",negative,neutral
...,...,...,...
2132,moved into encoders/decoders,neutral,neutral
2133,"I understand. In that case, we would need to h...",negative,negative
2134,that is a good idea.,positive,positive
2135,"Just out of curiosity, why `uv_buf_init` shoul...",neutral,neutral


In [13]:
df_premo['deva_polarity'] = df_premo['valence'].apply(valence_to_polarity)
df_premo = df_premo.drop(columns=['valence'], errors='ignore')
df_premo

,text,polarity,deva_polarity
0,did you get a chance to follow up on the issue...,neutral,neutral
1,Thank you for this PR. This is an interesting...,neutral,negative
2,", oh sorry for this, my fault, i'll be more c...",negative,negative
3,I've added so that we can add documentation r...,neutral,neutral
4,Applications can still use or qualifiers to g...,negative,negative
...,...,...,...
1786,"Hmm, looks like r737027 corresponds to version...",negative,neutral
1787,"<!-- First of all, thank you for your contribu...",neutral,negative
1788,"there was a typo in the comment, also made a s...",neutral,neutral
1789,Fixed some typos. Added the ghost button type....,neutral,negative


## Calculating precision, recall, F1 for DEVA results

For each class $cls$ of $classes=[positive, negative, neutral]$, we compute:
$$\text{precision}(cls) = \frac{TP}{TP + FP}\;\;\;\;\;\;\;\;\text{(of all messages predicted as class cls, the fraction that are truly cls)}$$
$$\text{recall}(cls) = \frac{TP}{TP + FN}\;\;\;\;\;\;\;\;\text{(of all messages that are truly of class cls, the fraction correctly identified)}$$
$$\text{f1}(cls) = 2\times\frac{\text{precision}(cls) \times \text{recall}(cls)}{\text{precision}(cls) + \text{recall}(cls)}\;\;\;\;\;\;\;\;\text{(harmonic mean of precision and recall)},$$
where for each class $cls$, TP is the number of messages correctly predicted as $cls$, FP is the number of messages incorrectly predicted as $cls$, and FN is the number of messages that belong to $cls$ but were predicted as a different class.

Moreover, for each metric, we compute macro averages and weighted averages:
$$\text{precision}_{\text{macro}} = \frac{1}{|classses|}\sum_{cls}\text{precision}(cls)$$
$$\text{precision}_{\text{weighted}} = \frac{\sum_{cls}\text{precision}(cls)\times|cls|}{\sum_{cls}|cls|}$$

We also aggregate TP, FP, FN across all classes to calculate an overall accuracy: micro-average. We use the following formulae:

$$\text{precision}_\text{micro}=\frac{\sum_{cls}TP_{cls}}{\sum_{cls}(TP_{cls}+FP_{cls}}$$
$$\text{recall}_\text{micro}=\frac{\sum_{cls}TP_{cls}}{\sum_{cls}(TP_{cls}+FN_{cls})}$$
$$\text{f1}_\text{micro}=2\times\frac{\text{precision}_\text{micro}\times \text{recall}_\text{micro}}{\text{precision}_\text{micro}+ \text{recall}_\text{micro}}$$

Note that in a multiclass setting where every sample gets exactly one label (as is in our case), $\sum TP_{cls} = \sum(TP_{cls} + FP_{cls}) = \sum(TP_{cls} + FN_{cls})$ (total number of samples). Therefore, $\text{precision}_\text{micro} = \text{recall}_\text{micro} = \text{f1}_\text{micro}$, which serves as an overall accuracy.

In [14]:
def compute_metrics(df, dataset_name, pred_col='deva_polarity', gold_col='polarity'):
    classes = ['positive', 'negative', 'neutral']
    rows = []

    for cls in classes:
        tp = ((df[pred_col] == cls) & (df[gold_col] == cls)).sum()
        fp = ((df[pred_col] == cls) & (df[gold_col] != cls)).sum()
        fn = ((df[pred_col] != cls) & (df[gold_col] == cls)).sum()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        rows.append({
            'dataset': dataset_name,
            'class': cls,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': (df[gold_col] == cls).sum()
        })


    # macro average (simple mean across classes, like Coutinho et al.)
    macro = pd.DataFrame(rows)
    rows.append({
        'dataset': dataset_name,
        'class': 'macro avg',
        'precision': macro['precision'].mean(),
        'recall': macro['recall'].mean(),
        'f1': macro['f1'].mean(),
        'support': len(df)
    })

    # weighted average (mean weighted by class support)
    total = len(df)
    rows.append({
        'dataset': dataset_name,
        'class': 'weighted avg',
        'precision': (macro['precision'] * macro['support']).sum() / total,
        'recall':    (macro['recall']    * macro['support']).sum() / total,
        'f1':        (macro['f1']        * macro['support']).sum() / total,
        'support': total
    })

    # micro average (aggregate TP/FP/FN across all classes first)
    tp_sum = sum(((df[pred_col] == c) & (df[gold_col] == c)).sum() for c in classes)
    fp_sum = sum(((df[pred_col] == c) & (df[gold_col] != c)).sum() for c in classes)
    fn_sum = sum(((df[pred_col] != c) & (df[gold_col] == c)).sum() for c in classes)
    micro_p = tp_sum / (tp_sum + fp_sum) if (tp_sum + fp_sum) > 0 else 0
    micro_r = tp_sum / (tp_sum + fn_sum) if (tp_sum + fn_sum) > 0 else 0
    micro_f = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0
    rows.append({
        'dataset': dataset_name,
        'class': 'micro avg',
        'precision': micro_p,
        'recall': micro_r,
        'f1': micro_f,
        'support': total
    })


    return pd.DataFrame(rows)

In [15]:
metrics_gold_standard = compute_metrics(df_gold_standard, 'gold_standard')
metrics_premo = compute_metrics(df_premo, 'premo')

metrics_all = pd.concat([metrics_gold_standard, metrics_premo], ignore_index=True)
metrics_all

,dataset,class,precision,recall,f1,support
0,gold_standard,positive,0.712733,0.745130,0.728571,616
1,gold_standard,negative,0.663235,0.718153,0.689602,628
2,gold_standard,neutral,0.801968,0.730123,0.764361,893
3,gold_standard,macro avg,0.725979,0.731135,0.727512,2137
4,gold_standard,weighted avg,0.735476,0.730931,0.732075,2137
5,gold_standard,micro avg,0.730931,0.730931,0.730931,2137
6,premo,positive,0.664903,0.723608,0.693015,521
7,premo,negative,0.484076,0.703704,0.573585,432
8,premo,neutral,0.798658,0.568019,0.663877,838
9,premo,macro avg,0.649212,0.665110,0.643492,1791


Note: the weighted average in the results is higher than the macro average mostly because the model seems to do much better on neutral text.

Note: the numbers are slightly different for the gs_gh set from the paper by Novielli et al. This is because I believe they run the experiemnts on the test set only (30% of the whole dataset), while I run it on the whole dataset. Moreover, the macro- and micro- averages are the same between both sets of results, confirming the validity of my replication.

Some takeaways:
- for the Github Golden Standard dataset, DEVA does relatively well, with a macro-average F1 score of 0.725. Neutral is the easiest class to predict (F1 score of 0.766), which makes sense given lexicon-based tools tend to default toward neutral when no strong sentiment words are found
- for the Coutinho dataset, DEVA struggles more: a macro-average F1 score of just 0.643. The difference is especially striking in the negative class performance, as the precision is very low (0.484). This means that DEVA is over-predicting negative sentiment on this dataset, flagging many non-negative items as negative. Interestingly, Coutinho et al. state that _"unlike positive messages, negative messages were frequently very subtle, sometimes even employing irony or sarcasm, which has been noted by previous works as a problem for these tools"_. However, this is not really the case? As based on the results, it seems like DEVA is very trigger-happy with negative labelling, possibly due to technical vocabulary. **This is something worth analyzing perhaps.**
- on the GH GS dataset, DEVA is conservative with neutral labels: it has high precision but moderate recall. However, on the coutinho dataset, it has high precision but low recall -- meaning it's missing a lot of neutral cases, likely identifying them as negative. In both cases, DEVA under-predicts neutral, but it is quite precise when it does assign neutral.

**Why does DEVA seem to be so precise with neutral labels, but also so conservative?**
It's mostly down to how DEVA works under the hood. DEVA computes the overall valence score of a text as whichever of the max positive score or max negative score has the larger absolute value. So, for a text to be classified as neutral, every sentiment-bearing word must score +- 1, or no words match the dictionary at all. Any word scoring +2 or -2 anywhere in the text will pull the whole document to positive or negative, regardless of the neutral context.

# LLM Experiments
We use three models: Gemma3:12b, Phi4:14b, and Llama3.1:8b.

In [16]:
# config

MODELS = ["gemma3:12b", "phi4:14b", "llama3.1:8b"]
CACHE_DIR = ROOT_DIR / "cache" / "llm"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MAX_CONSECUTIVE_ERRORS = 3

In [17]:
# cache helpers

def cache_path(model: str, prompt_id: int, dataset: str) -> Path:
    safe_name = model.replace(":", "_").replace("/", "_")
    return CACHE_DIR / f"{safe_name}_prompt_{prompt_id}_{dataset}.json"

def load_cache(model: str, prompt_id: int, dataset: str) -> dict:
    path = cache_path(model, prompt_id, dataset)
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            entries = json.load(f)
        return {e["text"]: (e.get("raw_message", ""), e["label"]) for e in entries}
    return {}

def save_cache(model: str, prompt_id: int, dataset: str, results: dict):
    entries = [
        {"text": text, "raw_message": raw, "label": label}
        for text, (raw, label) in results.items()
    ]
    with open(cache_path(model, prompt_id, dataset), "w", encoding="utf-8") as f:
        json.dump(entries, f, ensure_ascii=False, indent=1)

In [18]:
LABEL_PATTERN = re.compile(r'(?:Label|Sentiment):\s*(\w+)')

def extract_label(response: str) -> str:
    """Extract sentiment label from LLM response through the
    same processing step used in work by Zhang et al.

    For each line, first tries the regex pattern, then falls back to
    token-level substring matching. Defaults to 'neutral' if no
    sentiment keyword is found.

    This approach might not be optimal, but it is consistent
    with prior work.
    """
    for line in response.strip().lower().splitlines():
        line = line.strip()

        # regex match
        match = LABEL_PATTERN.search(line)
        if match:
            result = match.group(1).strip()
            if result in ('positive', 'negative', 'neutral'):
                return result

        # fallback: token-level substring check
        for token in line.split():
            if 'positive' in token:
                return 'positive'
            elif 'negative' in token:
                return 'negative'
            elif 'neutral' in token:
                return 'neutral'

    return 'neutral'

In [19]:
def classify(text: str, model: str, system_prompt: str) -> tuple[str, str]:
    resp = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Text: {text}"}
        ],
        options={"temperature": 0},
    )
    raw = resp.message.content
    return raw, extract_label(raw)

def classify_with_cache(texts, model: str, prompt_id: int, dataset: str, system_prompt: str) -> list[str]:
    cached = load_cache(model, prompt_id, dataset)
    results = dict(cached)  # copy
    labels = []
    consecutive_errors = 0

    for text in tqdm(texts, desc=f"{model} (prompt {prompt_id})"):
        if text in results:
            labels.append(results[text][1])
            consecutive_errors = 0
            continue

        try:
            raw, label = classify(text, model, system_prompt)
            results[text] = (raw, label)
            labels.append(label)
            consecutive_errors = 0
        except Exception as e:
            consecutive_errors += 1
            print(f"\n  Error on {model}: {e}")
            results[text] = ('', 'error')
            labels.append("error")

            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                print(f"  {MAX_CONSECUTIVE_ERRORS} consecutive errors: skipping {model}")
                labels.extend(["error"] * (len(texts) - len(labels)))
                break

        # save periodically (every 25 items)
        if len(labels) % 25 == 0:
            save_cache(model, prompt_id, dataset, results)

    save_cache(model, prompt_id, dataset, results)
    return labels

def run_classification(df, model: str, prompt_id: int, dataset: str, system_prompt: str):
    col = f"{model.replace(':', '_').replace('/', '_')}_prompt_{prompt_id}"
    df[col] = classify_with_cache(df["text"].tolist(), model, prompt_id, dataset, system_prompt)

For this analysis, we will be using the prompts of Zhang et al.

**Prompt 0:**

Please perform Sentiment Classification task. Given the sentence from GitHub pull request code review comments, assign a sentiment label from ['negative', 'neutral', 'positive']. Return label only without any other text.

**Prompt 1**

Please categorize the sentiment expressed in the following sentence from GitHub pull request code review comments as either (1) positive, (2) neutral, or (3) negative.

**Prompt 2**

I will give you a sentence from GitHub pull request code review comments. You need to reply with the sentiment expressed in the following sentence, either (1) positive, (2) neutral, or (3) negative.

In [20]:
PROMPT_0 ="""
Please perform Sentiment Classification task. Given the sentence from GitHub pull request code review comments, assign a sentiment label from ['negative', 'neutral', 'positive']. Return label only without any other text.
"""

PROMPT_1 ="""
Please categorize the sentiment expressed in the following sentence from GitHub pull request code review comments as either (1) positive, (2) neutral, or (3) negative.
"""

PROMPT_2 ="""
I will give you a sentence from GitHub pull request code review comments. You need to reply with the sentiment expressed in the following sentence, either (1) positive, (2) neutral, or (3) negative.
"""

PROMPTS = {0: PROMPT_0, 1: PROMPT_1, 2: PROMPT_2}

## Running LLMs

In [21]:
# work on copies so the original dataframes stay clean
df_gold_standard_llm = df_gold_standard[['text', 'polarity']].copy()
df_premo_llm = df_premo[['text', 'polarity']].copy()

In [22]:
# Gold standard
for model in MODELS:
    for prompt_id, prompt in PROMPTS.items():
        run_classification(
            df_gold_standard_llm,
            model,
            prompt_id,
            "gold_standard",
            prompt
        )

llama3.1:8b (prompt 2): 100%|██████████| 2137/2137 [00:00<00:00, 351278.71it/s]


In [23]:
# PRemo
for model in MODELS:
    for prompt_id, prompt in PROMPTS.items():
        run_classification(
            df_premo_llm,
            model,
            prompt_id,
            "premo",
            prompt
        )

llama3.1:8b (prompt 2): 100%|██████████| 1791/1791 [00:00<00:00, 351701.79it/s]


In [24]:
df_gold_standard_llm

,text,polarity,gemma3_12b_prompt_0,gemma3_12b_prompt_1,gemma3_12b_prompt_2,phi4_14b_prompt_0,phi4_14b_prompt_1,phi4_14b_prompt_2,llama3.1_8b_prompt_0,llama3.1_8b_prompt_1,llama3.1_8b_prompt_2
0,Guess there is a typo here: `translate`.,neutral,neutral,neutral,negative,neutral,negative,negative,negative,negative,neutral
1,"@arturoc: If you multiply-include `gst.h`, wou...",neutral,neutral,neutral,neutral,neutral,neutral,negative,negative,negative,neutral
2,Thank you Vlad. Your contribution to Mangos an...,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive
3,"@opdenkamp Hi Lars, I'm afraid that you forgot...",negative,neutral,neutral,neutral,negative,neutral,neutral,negative,neutral,negative
4,"Ok, so let's be paranoid.",negative,negative,neutral,neutral,neutral,negative,negative,negative,negative,neutral
...,...,...,...,...,...,...,...,...,...,...,...
2132,moved into encoders/decoders,neutral,neutral,neutral,neutral,neutral,neutral,neutral,positive,positive,neutral
2133,"I understand. In that case, we would need to h...",negative,negative,neutral,negative,neutral,neutral,negative,positive,neutral,neutral
2134,that is a good idea.,positive,positive,positive,positive,positive,positive,positive,positive,positive,positive
2135,"Just out of curiosity, why `uv_buf_init` shoul...",neutral,neutral,neutral,negative,negative,negative,negative,negative,negative,negative


In [25]:
df_premo_llm

,text,polarity,gemma3_12b_prompt_0,gemma3_12b_prompt_1,gemma3_12b_prompt_2,phi4_14b_prompt_0,phi4_14b_prompt_1,phi4_14b_prompt_2,llama3.1_8b_prompt_0,llama3.1_8b_prompt_1,llama3.1_8b_prompt_2
0,did you get a chance to follow up on the issue...,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral,neutral
1,Thank you for this PR. This is an interesting...,neutral,negative,neutral,negative,negative,negative,negative,negative,negative,negative
2,", oh sorry for this, my fault, i'll be more c...",negative,negative,negative,negative,positive,negative,negative,negative,negative,negative
3,I've added so that we can add documentation r...,neutral,neutral,neutral,neutral,positive,neutral,positive,positive,neutral,neutral
4,Applications can still use or qualifiers to g...,negative,negative,neutral,negative,negative,negative,negative,negative,negative,neutral
...,...,...,...,...,...,...,...,...,...,...,...
1786,"Hmm, looks like r737027 corresponds to version...",negative,neutral,neutral,neutral,positive,neutral,neutral,positive,positive,neutral
1787,"<!-- First of all, thank you for your contribu...",neutral,positive,positive,positive,positive,positive,positive,positive,positive,positive
1788,"there was a typo in the comment, also made a s...",neutral,negative,neutral,negative,negative,negative,negative,negative,negative,negative
1789,Fixed some typos. Added the ghost button type....,neutral,positive,positive,neutral,neutral,neutral,negative,negative,negative,neutral


In [26]:
# compute metrics for all model/prompt/dataset combinations
all_metrics = []

# DEVA
for ds_name, df in [('gold_standard', df_gold_standard), ('premo', df_premo)]:
    m = compute_metrics(df, ds_name, pred_col='deva_polarity', gold_col='polarity')
    m['model'] = 'DEVA'
    m['prompt'] = '—'
    all_metrics.append(m)

# LLMs
for ds_name, df in [('gold_standard', df_gold_standard_llm), ('premo', df_premo_llm)]:
    for model in MODELS:
        for prompt_id in PROMPTS:
            col = f"{model.replace(':', '_').replace('/', '_')}_prompt_{prompt_id}"
            m = compute_metrics(df, ds_name, pred_col=col, gold_col='polarity')
            m['model'] = model
            m['prompt'] = f'P{prompt_id}'
            all_metrics.append(m)

df_all_metrics = pd.concat(all_metrics, ignore_index=True)

In [28]:
# build comparison table: macro-avg P / R / F1 per tool, pivoted by dataset
macro = df_all_metrics[df_all_metrics['class'] == 'macro avg'].copy()

pivot = macro.pivot_table(
    index=['model', 'prompt'],
    columns='dataset',
    values=['precision', 'recall', 'f1']
)

# organise columns as (dataset, metric) in P / R / F1 order
pivot = pivot.swaplevel(axis=1)
metric_order = ['precision', 'recall', 'f1']
pivot = pivot[[
    (ds, m) for ds in ['gold_standard', 'premo'] for m in metric_order
]]
pivot.columns.names = ['dataset', 'metric']

# fixed row order: DEVA first, then each LLM with P0/P1/P2
row_order = [('DEVA', '\u2014')]
for model in MODELS:
    for p in range(len(PROMPTS)):
        row_order.append((model, f'P{p}'))
pivot = pivot.loc[row_order]

# style
f1_cols = [('gold_standard', 'f1'), ('premo', 'f1')]

# find best F1 overall per dataset (green)
best_overall = {col: pivot[col].max() for col in f1_cols}

# find best prompt F1 per LLM per dataset (yellow)
best_prompt = {}
for model in MODELS:
    model_rows = [(model, f'P{p}') for p in range(len(PROMPTS))]
    for col in f1_cols:
        best_val = pivot.loc[model_rows, col].max()
        for idx in model_rows:
            if pivot.loc[idx, col] == best_val:
                best_prompt[(idx, col)] = True

def highlight(data):
    styles = pd.DataFrame('', index=data.index, columns=data.columns)
    for col in f1_cols:
        for idx in data.index:
            if data.loc[idx, col] == best_overall[col]:
                styles.loc[idx, col] = 'background-color: #d4edda; font-weight: bold'
            elif (idx, col) in best_prompt:
                styles.loc[idx, col] = 'background-color: #fff3cd'
    return styles

pivot.style.format('{:.3f}').apply(lambda _: highlight(pivot), axis=None)